# **HiddenGems:** Analysing NBA contract value with machine learning.

**Module:** Final Year Project<br>
**Student:** Oisin Webb (K00293591)<br>

## **Profiling**
This notebook explores the players table loaded into PostgreSQL to find data quality issues before cleaning. The raw tables are left untouched, and cleaned versions are saved separately.

## **Cross-referencing source**
Values that look unusual are checked against: 
- **Basketball Reference**. https://www.basketball-reference.com/ 

In [1]:
# Author: Oisin Webb

import sys
sys.path.append("..")

from sqlalchemy import create_engine
from db_config import DB_URL

import pandas as pd

In [2]:
# Author: Oisin Webb
# Database connection
engine = create_engine(DB_URL)

## Profiling Players Table

In [3]:
# Author: Oisin Webb
# Reading in the players table in the .raw schema.

players_df = pd.read_sql("SELECT * FROM raw.players", engine)
players_df.shape

(5477, 10)

## What players_df.shape shows me
- 5477 rows in the table.
- 10 columns.

In [4]:
# Author: Oisin Webb
players_df.head(10)

,player_id,player_name,full_name,position,shoots,height_cm,weight_lbs,nickname,career_length_years,jersey_numbers
0,stirtbe01,Bennett Stirtz,Bennett Carl Stirtz,Point Guard,Right,193.0,186.0,NaN,NaN,NaN
1,maraad01,Aday Mara,Aday Mara,Center,Right,221.0,260.0,NaN,NaN,NaN
2,naberbo01,Bob Naber,Bob Naber,Forward,Right,190.0,185.0,NaN,1.0,{14}
3,traylro01,Robert Traylor,Robert Traylor,Center and Power Forward,Right,203.0,284.0,Tractor Traylor,7.0,"{54,34,32}"
4,volkoal01,Alexander Volkov,Alexander Volkov,Center,Right,208.0,218.0,Sasha,2.0,{8}
5,clarkco01,Coty Clarke,Coty Clarke,Small Forward,Right,201.0,232.0,NaN,1.0,{63}
6,flemiki01,Kingston Flemings,Kingston Flemings,Point Guard,Right,193.0,183.0,NaN,NaN,NaN
7,philola01,Labaron Philon Jr.,Labaron Dewayne Philon Jr.,Point Guard,Right,193.0,176.0,NaN,NaN,NaN
8,radzira01,Ray Radziszewski,Ray Radziszewski,Forward,Right,196.0,210.0,NaN,1.0,{19}
9,boothca01,Calvin Booth,Calvin Booth,Center,Left,211.0,230.0,NaN,10.0,{52}


## What players_df.head(10) shows me

- There are two name columns. Only `player_name` should be needed, because another table handles name differences.
- Some players have more than one position.
- `shoots` shows which hand the player shoots with.
- `nickname` is mostly empty, so it will be dropped in cleaning.
- `career_length_years` and `jersey_numbers` are empty for players picked in the 2026 NBA draft. They have no NBA seasons yet, so they are not used in this project.
- Players can have many jersey numbers during their careers.

In [5]:
# Author: Oisin Webb
players_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5477 entries, 0 to 5476
Data columns (total 10 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   player_id            5477 non-null   str    
 1   player_name          5477 non-null   str    
 2   full_name            5476 non-null   str    
 3   position             5476 non-null   str    
 4   shoots               5476 non-null   str    
 5   height_cm            5471 non-null   float64
 6   weight_lbs           5473 non-null   float64
 7   nickname             2139 non-null   str    
 8   career_length_years  4685 non-null   float64
 9   jersey_numbers       5373 non-null   str    
dtypes: float64(3), str(7)
memory usage: 795.8 KB


## What players_df.info() shows
- All the data types look acceptable, except `jersey_numbers`, which is stored as a string. This is because a player can wear several numbers over their career, so the column holds more than one value per player.

In [6]:
# Author: Oisin Webb
# Description of all the numeric columns

players_df.describe()

,height_cm,weight_lbs,career_length_years
count,5471.000000,5473.000000,4685.000000
mean,198.171632,209.665083,4.998506
std,9.139682,25.852960,4.470393
min,160.000000,114.000000,1.000000
25%,190.000000,190.000000,1.000000
50%,198.000000,210.000000,3.000000
75%,206.000000,225.000000,8.000000
max,231.000000,360.000000,22.000000


## What players_df.describe() shows me 
- 231 cm or 7'7 is the max in the `height_cm` it's not unheard of in the NBA.
- 360 pounds is the max in the `weight_lbs`.
- `career_length_years` min years is 1 which is understandable and max is 22 years.
### All above have been cross-referenced using basketball reference to verify stats are correct.

In [7]:
# Author: Oisin Webb
players_df[players_df["height_cm"] == 160][["player_name", "career_length_years","height_cm","weight_lbs"]]

,player_name,career_length_years,height_cm,weight_lbs
505,Penny Early,1.0,160.0,114.0
4894,Muggsy Bogues,14.0,160.0,136.0


- `Penny Early` and `Muggsy Bogues` were both indeed 160 cm or 5'3 according to basketball reference.

In [8]:
# Author: Oisin Webb
players_df[players_df["height_cm"] == 231][["player_name", "career_length_years","height_cm","weight_lbs"]]

,player_name,career_length_years,height_cm,weight_lbs
1972,Gheorghe Mureșan,6.0,231.0,303.0
4902,Manute Bol,10.0,231.0,200.0


- `Gheorghe Mureșan` and `Manute Bol` were both indeed 231 cm or 7'7 according to basketball reference.

In [9]:
# Author: Oisin Webb
players_df[players_df["weight_lbs"] == 114][["player_name","career_length_years","height_cm","weight_lbs"]]

,player_name,career_length_years,height_cm,weight_lbs
505,Penny Early,1.0,160.0,114.0


- `Penny Early` was indeed 114 pounds according to basketball reference.

In [10]:
# Author: Oisin Webb
players_df[players_df["weight_lbs"] == 360][["player_name","career_length_years","height_cm","weight_lbs"]]

,player_name,career_length_years,height_cm,weight_lbs
5448,Sim Bhullar,1.0,226.0,360.0


- `Sim Bhullar` was indeed 360 pounds according to basketball reference.

In [11]:
# Author: Oisin Webb
players_df[players_df["career_length_years"] == 22][["player_name","career_length_years","height_cm","weight_lbs"]]

,player_name,career_length_years,height_cm,weight_lbs
849,Vince Carter,22.0,198.0,220.0


- `Vince Carter` did indeed play 22 years in the NBA according to basketball reference.

In [12]:
# Author: Oisin Webb
players_df.isna().sum()

player_id                 0
player_name               0
full_name                 1
position                  1
shoots                    1
height_cm                 6
weight_lbs                4
nickname               3338
career_length_years     792
jersey_numbers          104
dtype: int64

In [13]:
# Author: Oisin Webb
players_df[players_df["full_name"].isna()]

,player_id,player_name,full_name,position,shoots,height_cm,weight_lbs,nickname,career_length_years,jersey_numbers
3386,vildolu01,L. Vildoza,NaN,NaN,NaN,NaN,NaN,NaN,NaN,{6}


In [14]:
# Author: Oisin Webb
players_df[players_df["height_cm"].isna()]

,player_id,player_name,full_name,position,shoots,height_cm,weight_lbs,nickname,career_length_years,jersey_numbers
47,nolenpa01,Paul Nolen,Paul Nolen,Center,Right,NaN,210.0,NaN,1.0,NaN
1907,mitchmu01,Murray Mitchell,Murray Mitchell,Center,Right,NaN,NaN,NaN,1.0,NaN
2746,wertira01,Ray Wertis,Ray Wertis,Guard,Right,NaN,NaN,NaN,2.0,"{18,14,8}"
2937,leedi01,Dick Lee,Dick Lee,Forward,Right,NaN,NaN,NaN,1.0,NaN
3386,vildolu01,L. Vildoza,NaN,NaN,NaN,NaN,NaN,NaN,NaN,{6}
3768,woodbo01,Bob Wood,Bob Wood,Guard,Right,NaN,175.0,NaN,1.0,{4}


## What the above shows me
- `player_name` has no blanks, so it is used instead of `full_name`.
- The 1 blank in `full_name`, `position` and `shoots` is one player, `vildolu01` ("L. Vildoza"). In cleaning his name becomes "Luca Vildoza" and his height and weight are filled from Basketball Reference.
- The other blank heights and weights are 1950s and 1960s players, outside the 2016 to 2026 seasons, so they are left blank.
- `nickname`, `career_length_years` and `jersey_numbers` have many gaps and are not needed, so they will be dropped.

In [15]:
# Author: Oisin Webb
players_df.nunique()

player_id              5477
player_name            5428
full_name              5427
position                 44
shoots                    2
height_cm                28
weight_lbs              149
nickname               1932
career_length_years      22
jersey_numbers         2585
dtype: int64

## What players_df.nunique() shows me

- `player_id` has 5,477 unique values, one per row, so it uniquely identifies each player and is used to link tables.
- `player_name` has 5,428 unique values, so 46 names are shared by different players. Names alone can't identify a player.
- `position` has 44 different values, more than basketball's 5 positions so these are counting the players that can or have played more than one position in their careers.


In [16]:
# Author: Oisin Webb
players_df.duplicated().sum()

np.int64(0)

- There are no duplicate rows, which matches every player_id being unique.

In [17]:
# Author: Oisin Webb
shared = players_df[players_df.duplicated("player_name", keep=False)].sort_values(["player_name","player_id"])
shared

,player_id,player_name,full_name,position,shoots,height_cm,weight_lbs,nickname,career_length_years,jersey_numbers
5004,bradlbi01,Bill Bradley,Bill Bradley,Small Forward and Shooting Guard,Right,196.0,205.0,"Dollar Bill, The Secretary of State, Mr. Presi...",10.0,{24}
5000,bradlbi02,Bill Bradley,Bill Bradley,Shooting Guard,Right,180.0,165.0,NaN,1.0,{34}
2447,smithbi01,Bill Smith,Bill Smith,Guard/Forward,Right,196.0,190.0,NaN,1.0,{16}
2462,smithwi01,Bill Smith,Bill Smith,Center,Right,213.0,220.0,NaN,2.0,{43}
1008,duffybo01,Bob Duffy,Bob Duffy,Forward,Right,193.0,175.0,Bob John,1.0,"{13,17}"
...,...,...,...,...,...,...,...,...,...,...
1497,jacksto02,Tony Jackson,Tony Jackson,Point Guard,Right,183.0,170.0,NaN,1.0,{23}
2334,mitchto02,Tony Mitchell,Tony Mitchell,Power Forward,Right,203.0,235.0,NaN,1.0,{9}
3469,mitchto03,Tony Mitchell,Tony Mitchell,Small Forward,Right,198.0,216.0,NaN,1.0,{18}
2333,russewa01,Walker Russell,Walker Russell,Point Guard,Right,196.0,195.0,NaN,6.0,"{33,2,31,23}"


- Checked the players that share a name as you can see the `player_id` column goes from `01` to `02` on the ID for players that share a name.
- Skipped `player_id` `mitchto01`. Will investigate below.

In [18]:
# Author: Oisin Webb
players_df[players_df["player_id"] == "mitchto01"]

,player_id,player_name,full_name,position,shoots,height_cm,weight_lbs,nickname,career_length_years,jersey_numbers
1900,mitchto01,Todd Mitchell,Todd Mitchell,Small Forward,Right,201.0,205.0,NaN,1.0,"{34,32}"


- `mitchto01` looks like it could be a third `Tony Mitchell` but it is `Todd Mitchell`, so he is not in the shared names list.
- `player_id` is a result of first 5 letters of the surname + first 2 letters of the first name + a number. The number counts players who share those same letters, which is why Todd and Tony both start with `mitchto`.


In [19]:
# Author: Oisin Webb
shared["player_name"].nunique()

46

## Investigating shared player names

- 46 names are shared by 95 players, but every `player_id` is unique (e.g. Tony Mitchell = mitchto02 and mitchto03).

In [20]:
# Author: Oisin Webb
players_df["position"].value_counts()

position
Shooting Guard                                                           835
Small Forward                                                            709
Center                                                                   689
Point Guard                                                              674
Power Forward                                                            602
Point Guard and Shooting Guard                                           219
Center and Power Forward                                                 208
Power Forward and Center                                                 188
Shooting Guard and Small Forward                                         171
Small Forward and Power Forward                                          168
Guard                                                                    166
Shooting Guard and Point Guard                                           164
Forward                                                            

- This is a mess and is not needed as my stats table contains a `pos` column that has the players positions listed properly and cleaner as PG,SG,SF,PF and C.
- I will be dropping the `position` column in this table during cleaning.

In [21]:
# Author: Oisin Webb
players_df["shoots"].value_counts()

shoots
Right    5092
Left      384
Name: count, dtype: int64

- As expected `shoots` only has right and left values with right handed shooters taking the vast majority.
- 1 blank value is `L. Vildoza` from above research.
- Keeping `shoots`. It is not used in the model but could be shown in the Streamlit app.

## Players table summary

| Column | Decision | Reason |
|---|---|---|
| player_id | Keep | Unique for every player, used to link tables |
| player_name | Keep | No blanks, used for name matching |
| full_name | Drop | Not needed, player_name is used instead |
| position | Drop | Messy (44 values), stats table `pos` is cleaner |
| shoots | Keep | Not used in the model but can be shown in the app |
| height_cm | Keep and fill | Fill Vildoza by hand, needed for the app height filter |
| weight_lbs | Keep and fill | Fill Vildoza by hand |
| nickname | Drop | Mostly empty and not needed |
| career_length_years | Drop | Many gaps and not needed |
| jersey_numbers | Drop | Not needed |

Also in cleaning, rename "L. Vildoza" to "Luca Vildoza".